#### Movements within parks (Albere, Gocciadoro & Solzenicyn)
Application of the functions defined in `trips_within_parks.py`  
Goal: investigate e-scooters movements within parks and computing their speed

In [1]:
from trips_within_parks import *

In [2]:
# get parks geometries
parchi = get_parks() # -> MultiPolygon

In [3]:
type(parchi)

shapely.geometry.multipolygon.MultiPolygon

In [4]:
# Gocciadoro
gocciadoro_park = parchi[0]
# Albere
albere_park = parchi[1]
# Solzenicy
solzenicy_park = parchi[2]

In [5]:
# dictionary with counts of trips intersecting each park (True)
count_trips_within_parks(parchi)

{'Gocciadoro': Counter({False: 87896, True: 27}),
 'Solzenicy': Counter({False: 85441, True: 2482}),
 'Albere': Counter({False: 86079, True: 1844})}

In [6]:
# dataset with trips passing through Albere park
routes_inside_albere = inside_park(albere_park)

In [15]:
# distinguish in: 
# - trips starting inside the park
# - trips ending inside the park
# - trips only passing through

print('Albere:')
stops_or_transit(routes_inside_albere, albere_park)

Albere:


{'tot_trips_within': 1839,
 'starting_within': 251,
 'ending_within': 493,
 'start_AND_end_within': 35,
 'transit_through_only': 1130}

In [16]:
print('Gocciadoro:')
routes_inside_gocciadoro = inside_park(gocciadoro_park)
print(stops_or_transit(routes_inside_gocciadoro, gocciadoro_park))

print('Solzenicy:')
routes_inside_Solzenicy = inside_park(solzenicy_park)
print(stops_or_transit(routes_inside_Solzenicy, solzenicy_park))

Gocciadoro:
{'tot_trips_within': 27, 'starting_within': 14, 'ending_within': 15, 'start_AND_end_within': 8, 'transit_through_only': 6}
Solzenicy:
{'tot_trips_within': 2482, 'starting_within': 22, 'ending_within': 45, 'start_AND_end_within': 3, 'transit_through_only': 2418}


In [17]:
# augment dataframe of trips passing through the Albere park, 
# with time of departure and arrival, route, travelled distance and speed
routes_inside_albere = compute_speed(albere_park)

In [18]:
plot_distrib_speed(routes_inside_albere, 'Velocità all\'interno del Parco delle Albere')

In [19]:
routes_inside_sol = compute_speed(solzenicy_park)
plot_distrib_speed(routes_inside_sol, 'Velocità all\'interno del Parco Solzenicy') # need to zoom-in with the lens (two weird outliers - cfr. below)

In [59]:
# TWO WEIRD OUTLIERS:
# these two values could not be matched by Valhalla (matched column values equal to False),
# and the speed are abnormally high, thus there are probably some errors in the coordinates
routes_inside_sol[routes_inside_sol.speed > 12] # speed > 12 m/s

,unique_id,point_timestamp,start_times,end_times,route,matched,geometry,pass_within,distance_meters,speed
2457,4d14b794acad_2021-02-28,"[2021-02-28 13:21:30+00:00, 2021-02-28 13:21:3...",2021-02-28 13:21:30+00:00,2021-02-28 13:21:34+00:00,"[[46.069654, 11.118777], [46.054284, 11.135467]]",False,"[[11.118777, 46.069654], [11.135467, 46.054284]]",True,2141.597792,535.399448
2470,a46a567ec9de_2021-07-06,"[2021-07-06 17:58:42+00:00, 2021-07-06 17:58:4...",2021-07-06 17:58:42+00:00,2021-07-06 17:58:44+00:00,"[[46.061417, 11.124776], [46.063546, 11.12582]]",False,"[[11.124776, 46.061417], [11.12582, 46.063546]]",True,250.051542,125.025771


In [20]:
routes_inside_gocciadoro = compute_speed(gocciadoro_park)
plot_distrib_speed(routes_inside_gocciadoro, 'Velocità all\'interno del Parco del Gocciadoro')

In [24]:
import pandas as pd
import plotly.express as px

In [38]:
albere_df = routes_inside_albere[['unique_id', 'speed']]
albere_df['park'] = 'Albere'
gocciadoro_df = routes_inside_gocciadoro[['unique_id', 'speed']]
gocciadoro_df['park'] = 'Gocciadoro'
sol_df = routes_inside_sol[['unique_id', 'speed']]
sol_df['park'] = 'Solzenicy'

In [53]:
parks_df = pd.concat([albere_df, gocciadoro_df, sol_df])

In [55]:
# remove outliers:
parks_df = parks_df[parks_df.speed <= 12]

# plot
fig = px.box(parks_df, x='park', y="speed", color="park",
            notched=True, 
            title="Boxplots of speed by park")
fig.show()

In [57]:
# using the function defined in trips_within_parks
parks_speed_boxplots(parchi)